In [23]:
import sys
print(sys.executable)

c:\Users\perna\anaconda3\envs\russek-lab\python.exe


In [24]:
import pandas as pd
import duckdb
import numpy as np
import matplotlib.pyplot as plt
print("all good")

all good


In [25]:
def parse_phh_file(filepath):
    """Parse a single .phh file and return a dict of its contents."""
    hand = {}
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if '=' in line:
                key, val = line.split('=', 1)
                hand[key.strip()] = val.strip()
    return hand

# Test it on one file
test_file = r"C:\Users\perna\OneDrive\Documents\poker_pluribushands_code\pluribus_phhdata\30\0.phh"
result = parse_phh_file(test_file)
for key, val in result.items():
    print(f"{key}: {val}")

variant: 'NT'
ante_trimming_status: true
antes: [0, 0, 0, 0, 0, 0]
blinds_or_straddles: [50, 100, 0, 0, 0, 0]
min_bet: 100
starting_stacks: [10000, 10000, 10000, 10000, 10000, 10000]
actions: ['d dh p1 3c9s', 'd dh p2 6d5s', 'd dh p3 9dTs', 'd dh p4 2sQs', 'd dh p5 AdKd', 'd dh p6 7cTc', 'p3 f', 'p4 f', 'p5 cbr 225', 'p6 f', 'p1 f', 'p2 f']
hand: 0
players: ['MrWhite', 'Gogo', 'Budd', 'Eddie', 'Bill', 'Pluribus']
finishing_stacks: [9950, 9900, 10000, 10000, 10150, 10000]


In [26]:
import ast
# parser
def parse_phh_file_full(filepath, session_id, hand_id):
    """
    Parse a single .phh file and return a list of rows,
    one row per pre-flop decision made by each player.
    """
    # Read raw fields
    raw = {}
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if '=' in line:
                key, val = line.split('=', 1)
                raw[key.strip()] = val.strip()

    # Parse the fields we need
    players = ast.literal_eval(raw['players'])
    starting_stacks = ast.literal_eval(raw['starting_stacks'])
    finishing_stacks = ast.literal_eval(raw['finishing_stacks'])
    actions = ast.literal_eval(raw['actions'])
    blinds = ast.literal_eval(raw['blinds_or_straddles'])

    # Extract hole cards per player from deal actions
    hole_cards = {}
    for action in actions:
        parts = action.split()
        if parts[0] == 'd' and parts[1] == 'dh':
            player_idx = int(parts[2][1:]) - 1  # p1 -> 0, p2 -> 1, etc.
            hole_cards[player_idx] = parts[3]

    # Extract pre-flop actions (stop when we hit a deal of community cards)
    preflop_actions = []
    for action in actions:
        parts = action.split()
        # community card deal signals end of preflop
        if parts[0] == 'd' and parts[1] == 'db':
            break
        # skip hole card deals
        if parts[0] == 'd':
            continue
        # this is a player action
        player_idx = int(parts[0][1:]) - 1
        action_type = parts[1]  # f = fold, cc = call, cbr = raise
        try:
            amount = int(parts[2]) if len(parts) > 2 else None
        except ValueError:
            amount = None    
        preflop_actions.append({
            'session': session_id,
            'hand': hand_id,
            'player_idx': player_idx,
            'player_name': players[player_idx],
            'action': action_type,
            'amount': amount,
            'finishing_stack': finishing_stacks[player_idx],
            'profit': finishing_stacks[player_idx] - starting_stacks[player_idx],
            'hole_cards': hole_cards.get(player_idx, None),
            'position': player_idx,
            'blind': blinds[player_idx],
        })

    return preflop_actions

# Test on same file
rows = parse_phh_file_full(test_file, session_id='30', hand_id=0)
for row in rows:
    print(row)

{'session': '30', 'hand': 0, 'player_idx': 2, 'player_name': 'Budd', 'action': 'f', 'amount': None, 'finishing_stack': 10000, 'profit': 0, 'hole_cards': '9dTs', 'position': 2, 'blind': 0}
{'session': '30', 'hand': 0, 'player_idx': 3, 'player_name': 'Eddie', 'action': 'f', 'amount': None, 'finishing_stack': 10000, 'profit': 0, 'hole_cards': '2sQs', 'position': 3, 'blind': 0}
{'session': '30', 'hand': 0, 'player_idx': 4, 'player_name': 'Bill', 'action': 'cbr', 'amount': 225, 'finishing_stack': 10150, 'profit': 150, 'hole_cards': 'AdKd', 'position': 4, 'blind': 0}
{'session': '30', 'hand': 0, 'player_idx': 5, 'player_name': 'Pluribus', 'action': 'f', 'amount': None, 'finishing_stack': 10000, 'profit': 0, 'hole_cards': '7cTc', 'position': 5, 'blind': 0}
{'session': '30', 'hand': 0, 'player_idx': 0, 'player_name': 'MrWhite', 'action': 'f', 'amount': None, 'finishing_stack': 9950, 'profit': -50, 'hole_cards': '3c9s', 'position': 0, 'blind': 50}
{'session': '30', 'hand': 0, 'player_idx': 1, '

In [27]:
import os
import duckdb
import pandas as pd
# data into duckdb
data_root = r"C:\Users\perna\OneDrive\Documents\poker_pluribushands_code\pluribus_phhdata"
all_rows = []

for session_folder in os.listdir(data_root):
    session_path = os.path.join(data_root, session_folder)
    if not os.path.isdir(session_path):
        continue
    for filename in os.listdir(session_path):
        if not filename.endswith('.phh'):
            continue
        hand_id = int(filename.replace('.phh', ''))
        filepath = os.path.join(session_path, filename)
        try:
            rows = parse_phh_file_full(filepath, session_id=session_folder, hand_id=hand_id)
            all_rows.extend(rows)
        except Exception as e:
            print(f"Error in {filepath}: {e}")

df = pd.DataFrame(all_rows)
print(f"Total rows: {len(df)}")
print(df.head())

Total rows: 61811
  session  hand  player_idx player_name action  amount  finishing_stack  \
0     100     0           2     MrWhite      f     NaN          10000.0   
1     100     0           3      MrPink    cbr   210.0           9790.0   
2     100     0           4     MrBrown      f     NaN          10000.0   
3     100     0           5    Pluribus      f     NaN          10000.0   
4     100     0           0      MrBlue     cc     NaN          10310.0   

   profit hole_cards  position  blind  
0     0.0       9c3d         2      0  
1  -210.0       Ah4h         3      0  
2     0.0       Th5s         4      0  
3     0.0       6c7s         5      0  
4   310.0       TcQc         0     50  


In [28]:
# saving into duckdb

db_path = r"C:\Users\perna\OneDrive\Documents\RussekLab\projects\poker-project1-RL\data\pluribus.duckdb"

con = duckdb.connect(db_path)
con.execute("DROP TABLE IF EXISTS preflop")
con.execute("CREATE TABLE preflop AS SELECT * FROM df")

print("saved to duckdb")

saved to duckdb


In [31]:
# Add true_session column
con.execute("""
    CREATE OR REPLACE TABLE preflop AS
    SELECT *,
        REGEXP_REPLACE(session, 'b$', '') as true_session
    FROM preflop
""")

# Add stack_at_decision using LAG on finishing_stack
con.execute("""
    CREATE OR REPLACE TABLE preflop AS
    SELECT *,
        COALESCE(
            LAG(finishing_stack) OVER (
                PARTITION BY true_session, player_name 
                ORDER BY hand
            ),
            10000
        ) as stack_at_decision
    FROM preflop
""")

con.close()
print("database built successfully")

database built successfully


In [ ]:
con = duckdb.connect(db_path)
result = con.execute("SELECT COUNT(*) as total_rows FROM preflop").df()
print(result)

result2 = con.execute("SELECT session, COUNT(*) as hands FROM preflop GROUP BY session ORDER BY session LIMIT 10").df()
print(result2)
con.close()

   total_rows
0       61811
  session  hands
0     100    434
1    100b    699
2     101    570
3    101b     19
4     102    442
5    102b    294
6     103    420
7    103b    692
8     104    681
9     105    221
